In [0]:
WITH quarter_end AS (
  SELECT
    fiscal_year,
    deployable_account_name,
    fiscal_year_quarter,
    usage_date,
    AVG(dbu_dollars_t28d_avg) AS dbu_dollars_t28d_avg
  FROM main.gtm_gold.account_consumption_daily
  WHERE horizontal_and_vertical_hierarchy_concatenated_emails LIKE CONCAT('%', :ae_email, '%')
    --AND deployable_account_name IN ('CNH Industrial', 'Iveco', 'EssilorLuxottica (see Luxottica)', 'Poste Italiane', 'Eni')
    AND fiscal_year in (2026,2027)
  GROUP BY fiscal_year, deployable_account_name, fiscal_year_quarter, usage_date
  QUALIFY ROW_NUMBER() OVER (PARTITION BY deployable_account_name, fiscal_year_quarter ORDER BY usage_date DESC) = 1

)

SELECT
  deployable_account_name,
  fiscal_year,
  fiscal_year_quarter,
  usage_date AS snapshot_date,
  ROUND(dbu_dollars_t28d_avg, 2) AS dbu_dollars_t28d_avg,
  ROUND(LAG(dbu_dollars_t28d_avg) OVER (PARTITION BY deployable_account_name ORDER BY usage_date), 2) AS prev_quarter_t28d_avg,
  ROUND(dbu_dollars_t28d_avg - LAG(dbu_dollars_t28d_avg) OVER (PARTITION BY deployable_account_name ORDER BY usage_date), 2) AS qoq_change,
  TRY_DIVIDE(
    dbu_dollars_t28d_avg - LAG(dbu_dollars_t28d_avg) OVER (PARTITION BY deployable_account_name ORDER BY usage_date),
    LAG(dbu_dollars_t28d_avg) OVER (PARTITION BY deployable_account_name ORDER BY usage_date)
  ) AS qoq_change_pct
FROM quarter_end
ORDER BY fiscal_year_quarter desc